### Install dependencies (run once)

In [8]:
# If you're in a clean environment, run this once, then restart the kernel if prompted.
# %pip install -U langgraph langchain langchain-core
# %pip install -U langchain-ollama  # preferred provider package for Ollama models

# If the import for langchain_ollama fails in your setup, fallback to:
# %pip install -U langchain-community

### Imports and model configuration

We import the core building blocks (LangGraph, prompts, parsers) and configure three Ollama-backed chat models:

Writer & Translator → llama3.1:8b
Validator → ministral-3:8b
We also set a small TEMPERATURE for more deterministic behavior.

In [9]:
from typing import TypedDict # for structured outputs
from langgraph.graph import StateGraph, START, END

# Prefer the dedicated Ollama provider
try:
    from langchain_ollama import ChatOllama
except ImportError:
    # Fallback if your environment only has the community package
    from langchain_community.chat_models import ChatOllama  # type: ignore

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

# ---- Models and runtime settings ----
MODEL_FR_EN = "llama3.1:8b"      # Writer + Translator
MODEL_VALIDATOR = "llama3.1:8b" # "ministral-3:8b"  # Validator
MAX_ATTEMPTS = 3
TEMPERATURE = 0.2  # low = more deterministic

def make_llm(model_name: str, temperature: float = TEMPERATURE) -> ChatOllama:
    """
    Create an Ollama-backed chat model.
    If Ollama runs on another host, pass base_url="http://<host>:11434".
    """
    return ChatOllama(model=model_name, temperature=temperature)

writer_llm = make_llm(MODEL_FR_EN)
translator_llm = make_llm(MODEL_FR_EN)
validator_llm = make_llm(MODEL_VALIDATOR)

### Define the shared state

LangGraph nodes communicate through a shared state dictionary.
We define the fields we'll pass between nodes: the French text, the English translation, validator feedback, approval flag, number of attempts, and an optional topic.

In [10]:
class GraphState(TypedDict, total=False):
    french_text: str
    english_text: str
    feedback: str
    approved: bool
    attempt_count: int
    topic: str


### Prompts and chains

We build three chains, one for each agent:

* **Writer**: Produces a short French note from a topic.
* **Translator**: Translates French → English, optionally using validator suggestions.
* **Validator**: Reviews translation quality and outputs strict JSON with `is_acceptable` and `suggestions`.

In [15]:
# 1) Writer (French)
writer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es un rédacteur français prolixe. "
     "Écris une note en français sur le sujet donné. Utilise un langage soutenu et complexe, difficile à traduire pour un anglophone. "
     "Évite les emojis et les caractères spéciaux autres que la ponctuation standard."),
    ("human",
     "Sujet: {topic}\nLongueur: 1-2 phrases.\n"
     "Rédige uniquement le texte en français, sans guillemets.")
])
writer_chain = writer_prompt | writer_llm | StrOutputParser()

# 2) Translator (English)
translator_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You translate French to English. "
     "You produce a very short English translation of the French input. "
     "If 'Suggestions' are provided, incorporate them and provide a better translation. "
     "Output only the translation."),
    ("human",
     "### French Text\n{french_text}\n\n"
     "### Suggestions for improvement (may be empty)\n{suggestions}\n\n"
     "Now provide the translation:")
])
translator_chain = translator_prompt | translator_llm | StrOutputParser()

# 3) Validator (JSON output)

validation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a rigorous bilingual reviewer. "
     "Judge whether the English translation is a good translation of the French source. "
     "Criteria: fidelity (no omissions/additions), lenght, fluency, tone/style, and correctness.\n"
     "Respond in STRICT JSON with exactly these keys:\n"
     "{{\n"
     '  "is_acceptable": true|false,\n'
     '  "suggestions": "detected errors and short actionable suggestions if false, else empty"\n'
     "}}\n"
     "Do not include any extra text."
    ),
    ("human",
     "### French Source\n{french_text}\n\n"
     "### English Translation\n{english_text}\n\n"
     "Is the translation acceptable? Provide suggestions if not.")
])
validator_parser = JsonOutputParser()
validator_chain = validation_prompt | validator_llm | validator_parser

### Node functions

Each node is a function that:

* **Writer**: Generates French text from the topic and initializes counters.
* **Translator**: Produces an English translation, using any feedback from the validator, and increments the attempt counter.
* **Validator**: Checks quality and returns approval + suggestions; robust to malformed JSON.

In [16]:
def write_french(state: GraphState) -> GraphState:
    """Create a short French text from the topic."""
    topic = state.get("topic") or "une brève mise à jour sur la journée"
    french = writer_chain.invoke({"topic": topic}).strip()
    print("\n[Writer] Texte français:\n", french, "\n")
    return {
        "french_text": french,
        "attempt_count": 0,
        "feedback": ""
    }

def translate_en(state: GraphState) -> GraphState:
    """Translate French to English using optional suggestions."""
    suggestions = state.get("feedback", "") or ""
    english = translator_chain.invoke({
        "french_text": state["french_text"],
        "suggestions": suggestions
    }).strip()
    attempts = int(state.get("attempt_count", 0)) + 1
    print(f"[Translator] Tentative {attempts} (EN):\n", english, "\n")
    return {"english_text": english, "attempt_count": attempts}

def validate(state: GraphState) -> GraphState:
    """Validate translation quality and provide suggestions if needed."""
    try:
        result = validator_chain.invoke({
            "french_text": state["french_text"],
            "english_text": state["english_text"]
        })
        approved = bool(result.get("is_acceptable", False))
        suggestions = (result.get("suggestions") or "").strip()
    except Exception as e:
        print("[Validator] Parsing error:", e)
        approved = False
        suggestions = "Return valid JSON. Ensure fidelity, fluency, tone, and correctness."
    print("[Validator] Approved:", approved)
    if not approved and suggestions:
        print("[Validator] Suggestions:\n", suggestions, "\n")
    return {"approved": approved, "feedback": "" if approved else suggestions}

### Conditional routing and graph compilation

We define a router that:

* If the translation isn't approved and attempts < MAX_ATTEMPTS, it retries the translator with feedback.
* Otherwise, it ends the workflow.

We then build and compile the LangGraph with nodes and edges.

In [17]:
def decide_next(state: GraphState):
    """Return the next node key or END directly."""
    attempts = int(state.get("attempt_count", 0))
    approved = bool(state.get("approved", False))
    if (not approved) and (attempts < MAX_ATTEMPTS):
        return "translate_en"  # go straight to the node
    return END

def build_graph():
    graph = StateGraph(GraphState)
    graph.add_node("write_french", write_french)
    graph.add_node("translate_en", translate_en)
    graph.add_node("validate", validate)

    graph.add_edge(START, "write_french")
    graph.add_edge("write_french", "translate_en")
    graph.add_edge("translate_en", "validate")

    # No mapping needed if decide_next returns a node key or END
    graph.add_conditional_edges("validate", decide_next)

    return graph.compile()

app = build_graph()

### Run the graph

Set an initial topic (or leave it empty to use the default).
We invoke the compiled app and display the final result, including whether the validator approved, how many attempts were used, and any feedback if it didn't pass.

In [18]:
# You can change the topic here, or leave None to use the default
initial_state: GraphState = {
    "attempt_count": 0,
    "topic": "pourquoi l'IA ne remplacera jamais la créativité humaine"
}

final_state = app.invoke(initial_state)

print("\n================= FINAL RESULT =================")
print("Français:", final_state.get("french_text", "").strip())
print("English :", final_state.get("english_text", "").strip())
print("Approved:", final_state.get("approved", False))
if not final_state.get("approved", False):
    print("\nValidator feedback:\n", final_state.get("feedback", "").strip())
print("Attempts:", final_state.get("attempt_count", 0))
print("=================================================\n")


[Writer] Texte français:
 La notion selon laquelle l'intelligence artificielle (IA) pourrait un jour remplacer la créativité humaine est une hypothèse dénuée de fondements scientifiques, car elle repose sur une méconnaissance profonde des capacités cognitives et des processus mentaux qui sous-tendent les productions artistiques. En effet, la créativité humaine est le fruit d'une complexité inextricable de facteurs psychologiques, neurologiques et socioculturels qui ne peuvent être réduits à une simple programmation algorithmique. 

[Translator] Tentative 1 (EN):
 The notion that artificial intelligence (AI) could one day replace human creativity is a hypothesis lacking scientific foundations. 

[Validator] Approved: False
[Validator] Suggestions:
 The original text mentions that the hypothesis is 'dénuée de fondements scientifiques', which translates to 'lacking scientific foundations' but loses some nuance. A more faithful translation would be: The notion that artificial intelligence

### Conclusion 
This LangGraph workflow demonstrates conditional routing based on validation feedback, enabling iterative improvement of translations through multiple attempts.

Now, it's your turn to experiment! Imagine a new use case where conditional routing could enhance the workflow. Perhaps a customer support scenario where responses are refined based on customer satisfaction feedback, or a content generation pipeline that iteratively improves drafts based on editorial reviews. Try building a LangGraph that incorporates these ideas, and see how conditional routing can help create more dynamic and responsive applications!